# Hybrid PIC: self-consistent fields and periodic boundaries

## Pablo Requeijo

This notebook corresponds to the third session of the HPC class. It analyzes the 1D hybrid particle-in-cell code: kinetic ions pushed with the Boris pusher, massless fluid electrons, and fields on a periodic Yee mesh.

## Problem context

The hybrid model (course slides 31–39) evolves the ions as macroparticles and closes the system with a massless electron fluid. In normalized units ($B_0 = n_0 = m_p = e = \mu_0 = 1$, lengths in $d_i$, times in $\Omega_{ci}^{-1}$, velocities in $v_A$):

$$\vec j = \nabla\times\vec B, \qquad \frac{\partial \vec B}{\partial t} = -\nabla\times\vec E, \qquad \vec E = -\vec v_e\times\vec B - \frac{\nabla P_e}{n} + \eta\,\vec j, \qquad \vec v_e = \vec u - \frac{\vec j}{n},\quad P_e = n T_e.$$

$\vec E$ is **not** evolved in time: it is recomputed from Ohm's law whenever the moments $n$, $\vec u$ (deposited from the particles with a linear shape function) or $\vec B$ change. Only $\vec B$ is advanced (Faraday).

**Grid**: 1D in $x$, periodic, Yee layout (slide 58): $E_x, B_y, B_z$ on dual nodes $x_{i+1/2}$, $E_y, E_z, B_x, n, \vec u$ on primal nodes $x_i$. Each dataset in `results.h5` carries a `centering` attribute.

**Time scheme**: iterated Crank–Nicolson predictor-corrector (as in PHARE). Predict $\vec B^*$ and $\vec E^*$ from step $n$, then, `iterations` times: push the particles from step $n$ with the averaged fields $(\vec E^n+\vec E^*)/2$, $(\vec B^n+\vec B^*)/2$, deposit the moments, and redo Faraday and Ohm. With 2 corrections the scheme is second order in $dt$; with 1 it is first order. The Catch2 test `[hybrid][convergence]` checks both (error ratios 4.00 and 2.0 when $dt$ halves).

**Periodic boundaries**: a particle leaving through $x = L$ re-enters at $x = 0$, and grid indices wrap the same way.

## How we verify correctness

1. **Periodic folding** (`periodic_drift`, test-particle mode): with $\vec E = 0.1\,\hat y$ and $\vec B = \hat z$, the guiding center drifts along $+x$ at $v_d = 0.1$. In a box of length $L = 4$ the trajectory must be the unbounded one folded into $[0, L)$.
2. **Eigenmode frequencies** (`alfven_R`, `alfven_L`): a circularly polarized parallel wave $b = B_y + iB_z = A\,e^{i(kx-\omega t)}$ in a cold plasma obeys $\omega^2 + k^2\omega - k^2 = 0$:
   $\omega_L = \tfrac{k^2}{2}\left(\sqrt{1+4/k^2}-1\right)$ (ion cyclotron, $>0$) and $\omega_R = -\tfrac{k^2}{2}\left(\sqrt{1+4/k^2}+1\right)$ (whistler, $<0$). The measured frequency must match it, and the energy must stay constant.
3. **Dispersion relation from noise** (`thermal_noise`): a warm plasma with no imposed wave. Particle noise excites every mode, and the $(k, \omega)$ spectrum of $b$ must follow both branches.

## Imports

In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt

NUMERIC_COLOR = "tab:blue"
ANALYTIC_COLOR = "0.2"
L_BRANCH_COLOR = "tab:blue"
R_BRANCH_COLOR = "tab:orange"

## Load the data

Run the executable first (from `PIC/build`):

```bash
./pic_prequeijo ../examples/periodic_drift.txt results.h5 /periodic_drift
./pic_prequeijo ../examples/alfven_R.txt results.h5 /alfven_R
./pic_prequeijo ../examples/alfven_L.txt results.h5 /alfven_L
./pic_prequeijo ../examples/thermal_noise.txt results.h5 /thermal_noise
```

In [ ]:
RESULTS = "build/results.h5"


def load_group(name):
    """Reads every dataset of one HDF5 group into a dict of arrays."""
    with h5py.File(RESULTS, "r") as f:
        return {key: f[name][key][()] for key in f[name]}


def omega_branch(k, branch):
    """Analytic cold-plasma frequency of the L (>0) or R (<0) branch."""
    k = np.asarray(k, dtype=float)
    root = np.sqrt(1.0 + 4.0 / k**2)
    if branch == "L":
        return 0.5 * k**2 * (root - 1.0)
    return -0.5 * k**2 * (root + 1.0)


drift = load_group("periodic_drift")
wave_R = load_group("alfven_R")
wave_L = load_group("alfven_L")
noise = load_group("thermal_noise")

## 1. Periodic boundaries

The guiding center of the unbounded motion is $x_{gc}(t) = x_0 + v_d t$ with $x_{gc}(0) = 2$ (the initial velocity $(1,0,0)$ is perpendicular to $\vec B$ and $\vec E \times \vec B$ is along $x$, so the gyration is centered on $x_0$ in $x$). Folded into $[0, 4)$, every sample must stay within one Larmor radius of the folded guiding center ($r_L = |\vec v_0 - \vec v_d| = 0.9$ in the drifting frame), and $x$ must never leave the box.

In [ ]:
L_BOX = 4.0
t, x, y = drift["t"], drift["x"], drift["y"]
x_gc = np.mod(2.0 + 0.1 * t, L_BOX)

# Distance on the periodic circle between each sample and the guiding center.
gap = np.abs(x - x_gc)
gap = np.minimum(gap, L_BOX - gap)
print(f"x range: [{x.min():.4f}, {x.max():.4f})  (box [0, {L_BOX}))")
print(f"boundary crossings: {np.sum(np.abs(np.diff(x)) > L_BOX / 2)}")
print(f"max |x - x_gc| on the circle: {gap.max():.4f} (Larmor radius 0.9)")

fig, (ax_t, ax_xy) = plt.subplots(1, 2, figsize=(11, 4))
ax_t.plot(t, x, ".", ms=1.5, color=NUMERIC_COLOR, label="x(t), numeric")
ax_t.plot(t, x_gc, ".", ms=1, color=ANALYTIC_COLOR,
          label="guiding center (analytic, folded)")
ax_t.set(xlabel=r"$t\ [\Omega_c^{-1}]$", ylabel="x", ylim=(0, L_BOX),
         title="x is wrapped into [0, L)")
ax_t.legend(loc="upper right", fontsize=8)

ax_xy.plot(x, y, ".", ms=1, color=NUMERIC_COLOR)
for edge in (0.0, L_BOX):
    ax_xy.axvline(edge, color=ANALYTIC_COLOR, lw=1, ls="--")
ax_xy.set(xlabel="x", ylabel="y", xlim=(-0.3, L_BOX + 0.3),
          title="Gyro-orbits re-enter through -x")
ax_xy.set_aspect("equal")
plt.tight_layout()
plt.show()

## 2. Circularly polarized eigenmodes

Space-time diagrams of $B_y(x, t)$: the R mode propagates towards $-x$ and the L mode towards $+x$. The dashed line marks the analytic phase velocity $\omega/k$.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=False)
for ax, (label, data) in zip(axes, [("R", wave_R), ("L", wave_L)]):
    t, xd, By = data["t"], data["x_dual"], data["By"]
    amplitude = np.abs(By).max()
    image = ax.pcolormesh(xd, t, By, cmap="RdBu_r", shading="auto",
                          vmin=-amplitude, vmax=amplitude)
    k = 2 * np.pi / (xd[-1] + (xd[1] - xd[0]) / 2)
    phase_speed = omega_branch(k, label) / k
    # A crest of By = A cos(kx - omega t) starts at x = 0 at t = 0; NaN
    # breaks the line where the folded crest wraps around the box.
    crest = np.mod(phase_speed * t, 2 * np.pi)
    crest[1:][np.abs(np.diff(crest)) > np.pi] = np.nan
    ax.plot(crest, t, color="black", lw=1.5, ls="--",
            label=fr"crest at $\omega/k = {phase_speed:.3f}$")
    ax.set(xlabel=r"$x\ [d_i]$", ylabel=r"$t\ [\Omega_{ci}^{-1}]$",
           title=f"{label} mode, $B_y(x,t)$")
    ax.legend(loc="upper right", fontsize=8)
    fig.colorbar(image, ax=ax, label=r"$B_y$")
plt.tight_layout()
plt.show()

The frequency is measured from the phase of the Fourier mode $\hat b_k(t) = \frac{1}{N}\sum_i b(x_i,t)\,e^{-ikx_i}$, which evolves as $e^{-i\omega t}$. The total energy $W = \sum (B^2/2)\,dx + \sum (n u^2/2)\,dx$ (the cold-ion kinetic energy is carried by the bulk flow) must stay constant.

In [ ]:
def measured_frequency(data):
    """Least-squares slope of the unwrapped mode phase: b_k ~ exp(-i w t)."""
    xd, t = data["x_dual"], data["t"]
    k = 2 * np.pi / (xd[-1] + (xd[1] - xd[0]) / 2)
    b = data["By"] + 1j * data["Bz"]
    mode = (b * np.exp(-1j * k * xd)).mean(axis=1)
    phase = np.unwrap(np.angle(mode))
    return k, -np.polyfit(t, phase, 1)[0], np.abs(mode)


def total_energy(data):
    """Magnetic plus bulk ion kinetic energy at every snapshot."""
    dx = data["x_primal"][1] - data["x_primal"][0]
    w_b = 0.5 * (data["Bx"]**2 + data["By"]**2 + data["Bz"]**2).sum(axis=1)
    u2 = data["ux"]**2 + data["uy"]**2 + data["uz"]**2
    w_k = 0.5 * (data["n"] * u2).sum(axis=1)
    return (w_b + w_k) * dx


fig, (ax_amp, ax_w) = plt.subplots(1, 2, figsize=(11, 3.5))
for label, data, color in [("R", wave_R, R_BRANCH_COLOR),
                           ("L", wave_L, L_BRANCH_COLOR)]:
    k, omega, amplitude = measured_frequency(data)
    exact = omega_branch(k, label)
    print(f"{label} mode, k d_i = {k:.3f}: omega = {omega:+.6f}, "
          f"analytic {exact:+.6f}, relative error {(omega - exact) / exact:.1e}")
    ax_amp.plot(data["t"], amplitude / amplitude[0], color=color,
                label=f"{label} mode")
    energy = total_energy(data)
    ax_w.plot(data["t"], energy / energy[0] - 1.0, color=color,
              label=f"{label} mode")
ax_amp.set(xlabel=r"$t\ [\Omega_{ci}^{-1}]$", ylabel=r"$|\hat b_k(t)| / |\hat b_k(0)|$",
           title="Mode amplitude")
ax_w.set(xlabel=r"$t\ [\Omega_{ci}^{-1}]$", ylabel=r"$W(t)/W(0) - 1$",
         title="Relative energy change")
for ax in (ax_amp, ax_w):
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 3. Dispersion relation from thermal noise

Fourier transform of $b = B_y + iB_z$ in space and time, with the convention $e^{i(kx - \omega t)}$: the L branch shows up at $\omega > 0$ (saturating at $\Omega_{ci}$) and the R (whistler) branch at $\omega < 0$ (growing like $k^2$). A Hann window in time limits spectral leakage. At large $|k|$ the numerical R branch sits slightly below the analytic curve: this is the grid dispersion of the centered differences, which see $k_{\rm eff} = (2/dx)\sin(k\,dx/2) < k$. The L branch broadens near $\Omega_{ci}$ because of cyclotron damping by the warm ions.

In [ ]:
t, xd = noise["t"], noise["x_dual"]
b = noise["By"] + 1j * noise["Bz"]
b = b - b.mean(axis=0)
window = np.hanning(len(t))[:, None]

dt_out = t[1] - t[0]
dx = xd[1] - xd[0]
# fft over x gives exp(-ikx); ifft over t gives exp(+i omega t), so a mode
# exp(i(k0 x - omega0 t)) peaks at (k0, omega0).
spectrum = np.fft.ifft(np.fft.fft(b * window, axis=1), axis=0)
power = np.fft.fftshift(np.abs(spectrum)**2)
k = np.fft.fftshift(np.fft.fftfreq(len(xd), dx)) * 2 * np.pi
omega = np.fft.fftshift(np.fft.fftfreq(len(t), dt_out)) * 2 * np.pi

fig, ax = plt.subplots(figsize=(7, 5))
image = ax.pcolormesh(k, omega, np.log10(power + power.max() * 1e-12),
                      cmap="magma", shading="auto")
k_curve = np.linspace(0.05, k.max(), 400)
for sign in (1, -1):
    ax.plot(sign * k_curve, omega_branch(k_curve, "L"), color="white", lw=1,
            ls="--", label="L branch (analytic)" if sign == 1 else None)
    ax.plot(sign * k_curve, omega_branch(k_curve, "R"), color="white", lw=1,
            ls=":", label="R branch (analytic)" if sign == 1 else None)
ax.set(xlabel=r"$k\ [d_i^{-1}]$", ylabel=r"$\omega\ [\Omega_{ci}]$",
       ylim=(-8, 3), xlim=(k.min(), k.max()),
       title=r"Power of $B_y + iB_z$ in $(k, \omega)$")
fig.colorbar(image, ax=ax, label=r"$\log_{10}$ power")
ax.legend(loc="lower left", fontsize=8)
plt.tight_layout()
plt.show()

## Expected result

- Periodic drift: $x \in [0, 4)$ at all times, several boundary crossings, every sample within $r_L = 0.9$ of the folded guiding center.
- R and L eigenmodes: measured $\omega$ within ~$10^{-3}$ of the analytic value at $k d_i = 1$ ($-1.618$ and $0.618$), constant amplitude and energy conserved to ~$10^{-10}$.
- Thermal noise: the spectral power lies on the analytic L and R branches.